# 03c · Ana model C üzerinde ablasyon ve duyarlılık (EĞİTİM, ~14 saat, kaldığı yerden devam eder)
Önce **05c** çalışmış olmalı (C, `final_c` aşamasına kayıtlı). Hiperparametreler C ile aynı; her varyant tek değişiklik.

| Sıra | Varyant | Seed | Soru |
|---|---|---|---|
| 1 | C: R sıcaklıktan bağımsız | 5 | Kazanç R(T)'den mi geliyor? (mekanizma) |
| 2 | C: eşit boyut (h=64) | 5 | Kazanç model boyutundan mı? (Hibrit ile eşit parametre) |
| 3 | C duyarlılık: ohmik yumuşatma yok / 5 s, W = 128 | 3 | Tasarım seçimlerine duyarlılık |

Ablasyon tablosunun diğer satırları 03b/03'ten gelir (+3 RC kolu, − omurgaya akım, − telafi = Hibrit, noI).
Süre: koşu başına ~30–60 dk → 1–2. bloklar ≈ 7 sa, 3. blok ≈ 7 sa. `RUN_SENS = False` ile 3. blok atlanabilir.

In [ ]:
import sys, os, json, time, warnings
import os; ROOT = os.path.abspath('..')  # repository root (run notebooks from the notebooks/ folder)
SMOKE = False                 # True: çok küçük hızlı deneme (kurulum testi)
sys.path.insert(0, ROOT)
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, torch
from data_utils import *
from models.experiment import *
from models import engine
exp = Experiment(ROOT, smoke=SMOKE)
exp.load_data()
print('Cihaz:', exp.dev, '| W =', exp.cfg['W'])

In [ ]:
RUN_SENS = True
sel = exp.load_selected(); abl_c, sens_c = c_specs(sel)
plan = [(n, s, sd, 'ablation_c') for n, s in abl_c.items() for sd in exp.cfg['FINAL_SEEDS']]
if RUN_SENS: plan += [(n, s, sd, 'sensitivity_c') for n, s in sens_c.items() for sd in exp.cfg['ABL_SEEDS']]
t0 = time.time()
for i, (name, spec, seed, stage) in enumerate(plan, 1):
    exp.run(name, spec, seed, stage, series=(name == 'C: eşit boyut (h=64)'), ablations=RC_ABLATIONS_TEST)
    print(f'[{i}/{len(plan)}] {name} s{seed} · geçen {(time.time() - t0) / 3600:.2f} sa')

In [ ]:
from scipy import stats
fc = exp.collect('final_c'); ab = exp.collect('ablation_c'); se = exp.collect('sensitivity_c') if (exp.p['runs'] / 'sensitivity_c').exists() else pd.DataFrame()
b3 = exp.collect('ablation_b'); a3 = exp.collect('ablation'); f2 = exp.collect('final')
D = pd.concat([fc, ab, se, b3[b3.model == 'RC + akım girişi'], a3[a3.model.isin(['RC: yalnız ohmik kol', 'Akımsız (noI)'])],
               f2[f2.model == 'Hibrit (GRU→Transformer)']], ignore_index=True)
D['k'] = np.where(D.temp == '-20degC', '−20°C', 'eğitim sıc.')
ORDER = [C_NAME, 'C: R sıcaklıktan bağımsız', 'C: eşit boyut (h=64)', 'RC + akım girişi', 'RC: yalnız ohmik kol', 'Hibrit (GRU→Transformer)', 'Akımsız (noI)',
         'C duyarlılık: ohmik yumuşatma yok', 'C duyarlılık: ohmik yumuşatma 5 s', 'C duyarlılık: W = 128']
sl = D.groupby(['model', 'k', 'seed'])[['rmse', 'bias']].mean() * 100
oz = sl.groupby(['model', 'k']).agg(['mean', 'std']); oz.columns = [f'{a}_{b}' for a, b in oz.columns]
oz['n_seed'] = sl.groupby(['model', 'k']).size(); oz = oz.reset_index()
oz['val'] = oz.model.map(D.groupby(['model', 'seed']).val_rmse.first().groupby('model').mean() * 100)
oz['model'] = pd.Categorical(oz.model, ORDER); oz = oz.sort_values(['k', 'model'])
print(oz.round(3).to_string(index=False))
def pair(a, bm, k):
    m = D[D.k == k]; s = m.groupby(['model', 'seed']).rmse.mean() * 100; c = m.groupby(['model', 'seed', 'temp', 'cycle']).rmse.mean() * 100
    x = c.loc[a]; y = c.loc[bm].reindex(x.index); d = (x - y).dropna()
    return {'karşılaştırma': f'C vs {bm}', 'kapsam': k, 'Δ (C − varyant)': s.loc[a].mean() - s.loc[bm].mean(),
            'Welch p (seed)': stats.ttest_ind(s.loc[a], s.loc[bm], equal_var=False).pvalue, 'C daha iyi': f'{(d < 0).sum()}/{len(d)}',
            'Wilcoxon p (eşleştirilmiş)': stats.wilcoxon(d).pvalue if len(d) > 1 and (d != 0).any() else np.nan}
sig = pd.DataFrame([pair(C_NAME, v, k) for v in ORDER[1:] if v in set(D.model) for k in ['−20°C', 'eğitim sıc.']])
print(sig.round(4).to_string(index=False))

In [ ]:
from models.report import rc_param_table
rcp = rc_param_table(exp.records('final_c') + exp.records('ablation_c'), exp.norm)
print(rcp.groupby('model').mean(numeric_only=True).drop(columns='seed').round(1).T.to_string())
with pd.ExcelWriter(exp.p['excel'] / '03c_anamodel_C_ablasyon.xlsx') as xw:
    oz.to_excel(xw, sheet_name='Özet', index=False); sig.to_excel(xw, sheet_name='Anlamlılık', index=False)
    rcp.to_excel(xw, sheet_name='R parametreleri', index=False); D.to_excel(xw, sheet_name='Ham satırlar', index=False)
print('✅', exp.p['excel'] / '03c_anamodel_C_ablasyon.xlsx')